# Poison Frogs, part 1: the one-shot kill

Replicates the transfer-learning experiment of Shafahi et al., *Poison Frogs! Targeted Clean-Label Poisoning Attacks on Neural Networks* (NeurIPS 2018), on a dog vs. not-dog dataset. The concepts are explained in `README.md`.

* **Victim:** an ImageNet ResNet-50, frozen, with only its last layer (2,048 → dog / not dog) retrained.
* **Target:** one photo (`me.jpg`) that the clean model calls "not dog".
* **Poisons:** dog photos with a small pixel change, honestly labeled "dog" and added to training.

Run `python run.py prepare` once before this notebook. All the code lives in the `poisonfrogs` package; this notebook just calls it step by step.

In [ ]:
from IPython.display import Image, display

from poisonfrogs import config, plots, transfer

print("device:", config.DEVICE)
s = transfer.load_setup()          # frozen ResNet-50, cached features, and the target photo
print("train features:", tuple(s.X_train.shape), "  test features:", tuple(s.X_test.shape))

## 1. The attack

`transfer.run` does the whole part 1 experiment:

1. trains the clean last layer and asks it about the target,
2. picks the 5 training dogs whose features are already closest to the target,
3. crafts one poison from each with Algorithm 1 (`attack.craft`) and saves them as PNGs,
4. reads the poisons back like a victim would, labels them "dog", and retrains the last layer with 1 to 5 of them.

In [ ]:
r = transfer.run(s)

## 2. Before the attack

In [ ]:
display(Image(filename=str(plots.clean_prediction(s, r)), width=260))

## 3. The poisons: base dogs, poisons, and the change amplified 10x

In [ ]:
display(Image(filename=str(plots.poisons_grid(r)), width=900))

## 4. After retraining: targeted, not global

The target flips to dog; other faces in the test set don't.

In [ ]:
display(Image(filename=str(plots.targeted(s, r)), width=900))

## 5. What happened in feature space

A 2-D projection of the 2,048-d features. Arrows go from each base dog to its poison.

In [ ]:
display(Image(filename=str(plots.feature_space(s, r)), width=520))

## 6. Summary

In [ ]:
print(f"{'':32s}{'test acc':>10s}{'P(dog | target)':>18s}")
print(f"{'clean':32s}{r.clean_acc:>10.1%}{r.clean_p:>18.1%}")
for k, acc, p in r.dose:
    print(f"{f'+ {k} poison(s)':32s}{acc:>10.1%}{p:>18.1%}")

## 7. One poison, pixel by pixel

Made by `python run.py animate` (about a minute). Shown here if it exists.

In [ ]:
from IPython.display import HTML

gif = config.FIGURES / "pixel_shift.gif"
# linked, not embedded, so the notebook file stays small
display(HTML(f'<img src="{gif.relative_to(config.ROOT)}" width="900">')) if gif.exists() else print("run: python run.py animate")

## Takeaways

* **Clean label:** every poison is a real dog photo with a correct label, so a human reviewer has nothing to flag.
* **Targeted:** accuracy doesn't drop and other faces are unaffected. Only the chosen photo flips.
* **Limits:** this targets one exact image, and it needs the extractor's weights. Part 2 (`python run.py e2e-attack`, `e2e-ablation`) shows it gets much harder when the victim retrains every layer.